<p align="left">
  <img src="https://miro.medium.com/v2/resize:fit:1358/1*bSLNlG7crv-p-m4LVYYk3Q.png" width="360" style="display:inline-block;"/>
  <img src="https://play-lh.googleusercontent.com/02-8ZoVumyq5XGnyqz4ftm9lAJ0RNHTr5wHpPqSZjDedePNC-UmhAQDGh_sMsucmrqY=w3840-h2160-rw" width="420" style="display:inline-block;"/>
</p>

# Detecção de LIBRAS com YOLO

Projeto final da disciplina de **Ciência de Dados** — treinamento de um modelo YOLO para detecção do **alfabeto manual em LIBRAS** (Língua Brasileira de Sinais).

**Classe inédita:** Os 26 sinais manuais de LIBRAS **não estão** entre as 80 classes do COCO, atendendo ao requisito do projeto.

## Etapas
1. Instalação de pacotes
2. Download do dataset (Roboflow)
3. Exploração e visualização das anotações
4. Treinamento do YOLO
5. Avaliação (mAP, precisão, recall, matriz de confusão)
6. Inferência em imagens reais capturadas pela equipe

> ⚠️ Este notebook é projetado para rodar no **Google Colab com GPU** (Runtime → Change runtime type → T4 GPU).

---
## 1. Pacotes e Verificação do Ambiente

In [ ]:
!pip install ultralytics roboflow --quiet
import ultralytics
ultralytics.checks()

In [ ]:
import os
import glob
import random
import yaml
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from collections import Counter

---
## 2. Download do Dataset

Usaremos um dataset público de **alfabeto manual em LIBRAS** disponível no [Roboflow Universe](https://universe.roboflow.com/).

### Como obter o dataset:
1. Acesse https://universe.roboflow.com/ e pesquise por `LIBRAS` ou `brazilian sign language alphabet`.
2. Escolha um dataset com formato **YOLOv8** e divisão train/val/test.
3. Crie uma conta gratuita no Roboflow e copie sua **API key** em https://app.roboflow.com/settings/api.
4. Cole o snippet de download fornecido pelo Roboflow (botão "Download Dataset" → formato YOLOv8) na célula abaixo.

Exemplos de datasets que servem:
- `libras-alphabet` (variações disponíveis)
- `lingua-de-sinais` (vários workspaces)
- Buscar por "sign language" e filtrar por LIBRAS

> Se o dataset escolhido tiver < 1000 imagens, considere fazer **data augmentation** no próprio Roboflow antes do download.

In [ ]:
# ===== SUBSTITUA pelo snippet do Roboflow do dataset que você escolher =====
from roboflow import Roboflow
rf = Roboflow(api_key="SUA_API_KEY_AQUI")
project = rf.workspace("WORKSPACE_NAME").project("PROJECT_NAME")
version = project.version(1)
dataset = version.download("yolov8")

DATASET_PATH = dataset.location
print(f"Dataset baixado em: {DATASET_PATH}")

### Estrutura esperada após download

```
<DATASET_PATH>/
├── train/
│   ├── images/
│   └── labels/
├── valid/
│   ├── images/
│   └── labels/
├── test/
│   ├── images/
│   └── labels/
└── data.yaml
```

In [ ]:
# Verifica e mostra o conteúdo do data.yaml
yaml_path = os.path.join(DATASET_PATH, 'data.yaml')
with open(yaml_path, 'r') as f:
    data_cfg = yaml.safe_load(f)

print("Configuração do dataset:")
print(yaml.dump(data_cfg, sort_keys=False, allow_unicode=True))

nomes_classes = data_cfg['names']
num_classes = data_cfg.get('nc', len(nomes_classes))
print(f"Número de classes: {num_classes}")
print(f"Classes: {nomes_classes}")

---
## 3. Exploração do Dataset

Antes de treinar, é importante visualizar exemplos e checar o balanceamento entre classes.

In [ ]:
# Distribuição das classes no conjunto de treino
train_labels_dir = os.path.join(DATASET_PATH, 'train', 'labels')

contagem = Counter()
for label_file in glob.glob(os.path.join(train_labels_dir, '*.txt')):
    with open(label_file) as f:
        for line in f:
            class_id = int(line.split()[0])
            contagem[nomes_classes[class_id]] += 1

# Plot do balanceamento
fig, ax = plt.subplots(figsize=(14, 5))
classes_ordenadas = sorted(contagem.keys())
valores = [contagem[c] for c in classes_ordenadas]
ax.bar(classes_ordenadas, valores, color='steelblue')
ax.set_xlabel('Classe (letra)')
ax.set_ylabel('Número de instâncias')
ax.set_title('Distribuição de classes no conjunto de treino')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nTotal de instâncias: {sum(valores)}")
print(f"Classe com mais exemplos: {max(contagem, key=contagem.get)} ({max(valores)})")
print(f"Classe com menos exemplos: {min(contagem, key=contagem.get)} ({min(valores)})")

In [ ]:
# Visualizar imagens aleatórias do conjunto de teste com suas bounding boxes
test_img_dir = os.path.join(DATASET_PATH, 'test', 'images')
test_label_dir = os.path.join(DATASET_PATH, 'test', 'labels')

image_files = [f for f in os.listdir(test_img_dir) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
sample_images = random.sample(image_files, min(6, len(image_files)))

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, img_file in enumerate(sample_images):
    img_path = os.path.join(test_img_dir, img_file)
    label_path = os.path.join(test_label_dir, os.path.splitext(img_file)[0] + '.txt')

    img = Image.open(img_path).convert("RGB")
    draw = ImageDraw.Draw(img)
    w, h = img.size
    font_size = max(int(w * 0.05), 14)

    if os.path.exists(label_path):
        with open(label_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                class_id = int(parts[0])
                xc, yc, bw, bh = map(float, parts[1:5])
                x1 = (xc - bw/2) * w
                y1 = (yc - bh/2) * h
                x2 = (xc + bw/2) * w
                y2 = (yc + bh/2) * h
                draw.rectangle([x1, y1, x2, y2], outline='lime', width=4)
                classe = nomes_classes[class_id] if class_id < len(nomes_classes) else str(class_id)
                draw.text((x1, max(0, y1 - font_size)), classe, fill='lime')

    axes[i].imshow(img)
    axes[i].axis('off')
    axes[i].set_title(f'Exemplo {i+1}')

plt.tight_layout()
plt.savefig('test_samples.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Treinamento do Modelo YOLO

### Escolha da variante:
| Variante | Parâmetros | Quando usar |
|---|---|---|
| `yolov8n.pt` (nano) | 3.2M | Testes rápidos, GPU fraca |
| `yolov8s.pt` (small) | 11.2M | Bom equilíbrio velocidade/qualidade |
| `yolov8m.pt` (medium) | 25.9M | **Recomendado** para este projeto |
| `yolov8l.pt` (large) | 43.7M | Se tiver GPU robusta |
| `yolov8x.pt` (xlarge) | 68.2M | Máxima qualidade, treino lento |

### Hiperparâmetros principais:
- `epochs`: 50–100 (começar com 50 e monitorar overfitting)
- `batch`: 16–32 dependendo da memória da GPU (T4 = 16, A100 = 32+)
- `imgsz`: 640 (padrão YOLO)
- `patience`: 15 (early stopping)

Documentação oficial: https://docs.ultralytics.com/pt/modes/train/

In [ ]:
# Liberar memória da GPU antes do treino
import gc
import torch
gc.collect()
torch.cuda.empty_cache()

In [ ]:
from ultralytics import YOLO

# Carrega o modelo pré-treinado YOLOv8m
model = YOLO('yolov8m.pt')

# Treinamento
results_train = model.train(
    data=os.path.join(DATASET_PATH, 'data.yaml'),
    epochs=50,
    imgsz=640,
    batch=16,
    patience=15,
    name='libras_yolov8m',
    project='runs/detect',
    exist_ok=True,
    save=True,
    plots=True,
)

### Visualização dos gráficos de treinamento

O Ultralytics gera automaticamente plots em `runs/detect/libras_yolov8m/`. Vamos visualizar os principais.

In [ ]:
train_dir = model.trainer.save_dir if hasattr(model, 'trainer') else 'runs/detect/libras_yolov8m'
print(f"Diretório do treino: {train_dir}")

# Mostrar a imagem de resultados (curvas de loss, mAP, etc)
results_img = os.path.join(train_dir, 'results.png')
if os.path.exists(results_img):
    img = Image.open(results_img)
    plt.figure(figsize=(18, 10))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Curvas de treinamento')
    plt.show()

In [ ]:
# Exemplo de data augmentation aplicada durante o treino
aug_img = os.path.join(train_dir, 'train_batch0.jpg')
if os.path.exists(aug_img):
    img = Image.open(aug_img)
    plt.figure(figsize=(12, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Batch de treino com data augmentation')
    plt.show()

---
## 5. Avaliação do Modelo

Métricas exigidas pelo projeto:
- **mAP@0.5**: precisão média com IoU ≥ 0.5
- **mAP@0.5:0.95**: média do mAP para IoU de 0.5 a 0.95
- **Precisão (Precision)**: TP / (TP + FP)
- **Revocação (Recall)**: TP / (TP + FN)

In [ ]:
# Avaliação no conjunto de TESTE
results_val = model.val(
    data=os.path.join(DATASET_PATH, 'data.yaml'),
    split='test',
    plots=True,
)

In [ ]:
# Métricas globais
metrics = results_val.results_dict
print("=" * 50)
print("📊 MÉTRICAS GLOBAIS NO CONJUNTO DE TESTE")
print("=" * 50)
print(f"Precisão média:   {metrics.get('metrics/precision(B)', 0):.4f}")
print(f"Revocação média:  {metrics.get('metrics/recall(B)', 0):.4f}")
print(f"mAP@0.5:          {metrics.get('metrics/mAP50(B)', 0):.4f}")
print(f"mAP@0.5:0.95:     {metrics.get('metrics/mAP50-95(B)', 0):.4f}")
print("=" * 50)

In [ ]:
# Métricas por classe
import pandas as pd

per_class_data = []
for i, nome in enumerate(nomes_classes):
    try:
        p = results_val.box.p[i] if i < len(results_val.box.p) else None
        r = results_val.box.r[i] if i < len(results_val.box.r) else None
        map50 = results_val.box.ap50[i] if i < len(results_val.box.ap50) else None
        map5095 = results_val.box.ap[i] if i < len(results_val.box.ap) else None
        per_class_data.append({
            'Classe': nome,
            'Precisão': round(float(p), 4) if p is not None else None,
            'Recall': round(float(r), 4) if r is not None else None,
            'mAP@0.5': round(float(map50), 4) if map50 is not None else None,
            'mAP@0.5:0.95': round(float(map5095), 4) if map5095 is not None else None,
        })
    except (IndexError, AttributeError):
        continue

df_metricas = pd.DataFrame(per_class_data)
display(df_metricas)
df_metricas.to_csv('metricas_por_classe.csv', index=False)

In [ ]:
# Matriz de confusão (gerada pelo Ultralytics em runs/detect/val/)
val_dir = results_val.save_dir if hasattr(results_val, 'save_dir') else 'runs/detect/val'
conf_matrix_img = os.path.join(val_dir, 'confusion_matrix.png')
conf_matrix_norm_img = os.path.join(val_dir, 'confusion_matrix_normalized.png')

for path, titulo in [(conf_matrix_img, 'Matriz de Confusão'),
                      (conf_matrix_norm_img, 'Matriz de Confusão Normalizada')]:
    if os.path.exists(path):
        img = Image.open(path)
        plt.figure(figsize=(14, 12))
        plt.imshow(img)
        plt.axis('off')
        plt.title(titulo)
        plt.show()

In [ ]:
# Curvas PR, F1, P e R
for arq, titulo in [
    ('PR_curve.png', 'Curva Precisão x Revocação'),
    ('F1_curve.png', 'Curva F1 x Confiança'),
    ('P_curve.png', 'Curva Precisão x Confiança'),
    ('R_curve.png', 'Curva Revocação x Confiança'),
]:
    path = os.path.join(val_dir, arq)
    if os.path.exists(path):
        img = Image.open(path)
        plt.figure(figsize=(10, 7))
        plt.imshow(img)
        plt.axis('off')
        plt.title(titulo)
        plt.show()

---
## 6. Inferência em Imagens Reais

### Como subir suas fotos no Colab:
1. Capture fotos suas fazendo letras em LIBRAS com o celular.
2. Use o menu lateral do Colab (ícone de pasta) → upload para `/content/`.
3. Ou rode a célula `files.upload()` abaixo.

Dicas para boas fotos:
- Fundo neutro (parede branca, lousa)
- Iluminação frontal uniforme
- Mão centralizada e visível por inteiro
- Distância de 30–80 cm da câmera

In [ ]:
# Opção A: upload manual via Colab
from google.colab import files
uploaded = files.upload()
user_images = [f'/content/{name}' for name in uploaded.keys()]
print(f"Imagens carregadas: {user_images}")

In [ ]:
# Inferência nas imagens reais + algumas do conjunto de teste
test_samples = random.sample(
    glob.glob(os.path.join(DATASET_PATH, 'test', 'images', '*.jpg')),
    min(3, len(glob.glob(os.path.join(DATASET_PATH, 'test', 'images', '*.jpg'))))
)

todas_imagens = user_images + test_samples

preds = [model.predict(img, conf=0.25)[0] for img in todas_imagens]

# Plotar predições
n = len(preds)
cols = min(3, n)
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(cols * 6, rows * 5))
if rows == 1 and cols == 1:
    axes = [axes]
else:
    axes = axes.flatten() if hasattr(axes, 'flatten') else axes

for i, pred in enumerate(preds):
    img_bgr = pred.plot()
    img_rgb = img_bgr[:, :, ::-1]
    origem = 'REAL (equipe)' if i < len(user_images) else 'TESTE (dataset)'
    axes[i].imshow(img_rgb)
    axes[i].set_title(f'{origem} - {os.path.basename(todas_imagens[i])}')
    axes[i].axis('off')

for j in range(len(preds), len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.savefig('predicoes_reais.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Detalhes das detecções: classe, confiança, coordenadas
print("=" * 70)
print("DETALHES DAS DETECÇÕES")
print("=" * 70)
for i, pred in enumerate(preds):
    origem = 'REAL' if i < len(user_images) else 'TESTE'
    print(f"\n[{origem}] {os.path.basename(todas_imagens[i])}")
    if len(pred.boxes) == 0:
        print("   ⚠️ Nenhuma detecção")
        continue
    for box in pred.boxes:
        cls_id = int(box.cls[0])
        conf = float(box.conf[0])
        xyxy = box.xyxy[0].tolist()
        print(f"   → {nomes_classes[cls_id]:>3s}  conf={conf:.3f}  bbox=[{xyxy[0]:.0f}, {xyxy[1]:.0f}, {xyxy[2]:.0f}, {xyxy[3]:.0f}]")

---
## 7. Exportação dos Artefatos

Baixa o modelo treinado e os principais artefatos para colocar no repositório do GitHub.

In [ ]:
import shutil

# Pasta para empacotar tudo
os.makedirs('/content/artefatos', exist_ok=True)

# best.pt
best_pt = os.path.join(train_dir, 'weights', 'best.pt')
if os.path.exists(best_pt):
    shutil.copy(best_pt, '/content/artefatos/best.pt')

# Plots do treino
for arq in ['results.png', 'results.csv', 'confusion_matrix.png',
            'confusion_matrix_normalized.png', 'PR_curve.png',
            'F1_curve.png', 'P_curve.png', 'R_curve.png', 'labels.jpg']:
    src = os.path.join(train_dir, arq)
    if os.path.exists(src):
        shutil.copy(src, f'/content/artefatos/{arq}')

# Plots da validação
for arq in ['confusion_matrix.png', 'confusion_matrix_normalized.png',
            'PR_curve.png', 'F1_curve.png']:
    src = os.path.join(val_dir, arq)
    dst = f'/content/artefatos/val_{arq}'
    if os.path.exists(src):
        shutil.copy(src, dst)

# Métricas e exemplos
for arq in ['metricas_por_classe.csv', 'class_distribution.png',
            'test_samples.png', 'predicoes_reais.png']:
    if os.path.exists(f'/content/{arq}'):
        shutil.copy(f'/content/{arq}', f'/content/artefatos/{arq}')

# Empacota tudo num zip
shutil.make_archive('/content/artefatos_libras', 'zip', '/content/artefatos')
print("✅ Artefatos prontos em /content/artefatos_libras.zip")
print("Faça o download e descompacte na pasta results/ do repositório.")

files.download('/content/artefatos_libras.zip')

---
## 8. Conclusão

Após o treinamento e avaliação, atualize:
1. O **README.md** do repositório com as métricas finais.
2. O **relatório técnico** (`report/relatorio_template.md`) com análise crítica:
   - Quais letras o modelo confunde mais? (olhe a matriz de confusão — letras visualmente parecidas como M/N, U/V tendem a confundir)
   - O modelo generalizou bem para suas fotos reais? Se não, por quê? (iluminação, fundo, distância)
   - O que faria diferente numa próxima iteração? (mais épocas, modelo maior, mais augmentation, dataset maior)

### Próximos passos sugeridos:
- Testar com YOLOv8l ou YOLOv11 para comparar
- Avaliar em vídeo (gesto contínuo, não estático)
- Tratar os sinais dinâmicos de LIBRAS (H, J, K, X, Y, Z) que envolvem movimento